# 05 · Mini GraphRAG

**Bloque:** 1:35–1:55 · **Práctica:** 1:40–1:55

### Teoría breve
- **GraphRAG:** RAG + **grafo de conocimiento**. En vez de solo buscar por similitud, usa **entidades y relaciones** explícitas.
- Útil para consultas **multi-salto (multi-hop)**, donde hay que conectar varios hechos.

Extraeremos relaciones del documento, construiremos un grafo y responderemos con él.

In [ ]:
import sys, os
REPO = os.path.abspath("..")
if REPO not in sys.path:
    sys.path.insert(0, REPO)
from config import get_chat_model, get_embeddings
print("Entorno cargado ✔")

## Paso 1 · Extraer entidades y relaciones
Usamos **salida estructurada** (`with_structured_output`) para obtener tripletas fiables (origen, relación, destino).

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class Relacion(BaseModel):
    origen: str = Field(description="Entidad de origen")
    relacion: str = Field(description="Relación entre las entidades")
    destino: str = Field(description="Entidad de destino")

class Grafo(BaseModel):
    relaciones: List[Relacion]

llm = get_chat_model()
texto = open(os.path.join(REPO, "data", "agentes_ia.md"), encoding="utf-8").read()

extractor = llm.with_structured_output(Grafo)
resultado = extractor.invoke(
    "Extrae las relaciones clave entre entidades (tecnologías, conceptos, empresas) "
    f"del siguiente texto. Sé conciso.\n\n{texto}"
)
triples = resultado.relaciones
print(f"{len(triples)} relaciones extraídas. Ejemplos:")
for r in triples[:5]:
    print(f"  {r.origen} --[{r.relacion}]--> {r.destino}")

## Paso 2 · Construir el grafo de conocimiento
> **TODO:** por cada tripleta, añade una arista de `r.origen` a `r.destino` con atributo `rel=r.relacion`.

In [ ]:
import networkx as nx

G = nx.DiGraph()
for r in triples:
    G.add_edge(r.origen, r.destino, rel=r.relacion)
print("Entidades:", G.number_of_nodes(), "| Relaciones:", G.number_of_edges())

## Paso 3 · Visualizar el grafo

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(11, 7))
pos = nx.spring_layout(G, seed=42, k=0.9)
nx.draw(G, pos, with_labels=True, node_color="#8B5CF6", font_size=8,
        node_size=1600, font_color="white", edge_color="#94a3b8", arrows=True)
nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, "rel"), font_size=7)
plt.title("Grafo de conocimiento del taller")
plt.axis("off")
plt.show()

## Paso 4 · Responder con el grafo (contexto estructurado)
Recuperamos las relaciones conectadas a una entidad y se las damos al LLM.

In [ ]:
def contexto_grafo(entidad: str) -> str:
    lineas = []
    for o, d, data in G.edges(data=True):
        if entidad.lower() in o.lower() or entidad.lower() in d.lower():
            lineas.append(f"{o} --{data['rel']}--> {d}")
    return "\n".join(lineas) or "(sin relaciones)"

ctx = contexto_grafo("RAG")
print("Relaciones encontradas:\n", ctx, "\n")
resp = llm.invoke(
    f"Usando SOLO estas relaciones, explica con qué se relaciona RAG y cómo.\n\n{ctx}"
).content
print(resp)

## ✅ Checkpoint 5 · ¡Cierre técnico!
Respondiste una pregunta usando **contexto estructurado** de un grafo.

**Recapitulando el taller:** LLM → Agente (LangChain) → Tools (MCP) → LangGraph → RAG → GraphRAG. 🎉